# 25 — Setup experiments: registers model, photo crops, preparation variants and controls

Thin notebook: it only **imports** and **calls** `src/registers_experiment.py`, `src/photo_crop_experiment.py`, `src/preprocess_variants.py` and `src/preprocess_controls.py`, then prints short summaries. It is the extraction side of the setup-selection round of 2026-09-30; the scores are computed by the evaluation pillar (`eVTOL-Visual-Evaluation/embedding_evaluation`: `scripts/registers_check.py`, `scripts/preprocess_variants_check.py`, `scripts/preprocess_controls_check.py`) and, for the crop test, by `scripts/photo_crop_experiment.py evaluate` and `scripts/photo_crop_registers.py evaluate report` in this repo.

**Question:** which model and which image preparation should the chapter use? Four experiments, in pipeline order:
1. **Registers model** (`facebook/dinov2-with-registers-large`, tag `dinov2-reg-large_518`): embeddings of the patent figures and the photos, CLS attention on the main sets, and the artifact-share table base vs registers.
2. **Photo crop test:** OWLv2 crops of each photo's aircraft, processed and embedded with the base model and with the registers model.
3. **Preparation variants:** drawings `thick1` / `thick2` (strokes grown by about 1 / 2 px at the model input) and the photo `grey` variant, registers model.
4. **Drawing controls:** `grey` (colour removed) and `contrast` (strokes darkened, not widened), the ink table and the five-column montage.

**Input:** notebooks 20–22 (patent tree: `processed/518`, `embeddings/`), notebook 24 (`EVTOLNEWS_DS`: `selection/sets/main.csv`, `processed/518`, base attention), notebook 23 (`view_state_experiments/exp3_main.csv`, `aircraft_common.csv`, for the montages).
**Output:** under `1639_LABELLED/` and `EVTOLNEWS_DS/`, listed in each step and checked in the last cell.

**Idempotent.** Extraction and attention skip a run whose saved figure set already matches; copies and processed images skip when their manifests list the same figures; the tables and montages skip when they exist and are newer than their inputs. With the outputs on disk the whole notebook runs in about ten seconds and writes nothing. `FORCE = True` rebuilds everything (detector, copies, processing, both GPUs for the extractions: several hours).

Kernel: Finetune env (`python3`), both GPUs (`analysis.devices`).

In [1]:
FORCE = False   # True: rebuild every step even when its outputs exist (hours, both GPUs)

## Setup
Loads `config.yaml`, pre-imports `AutoImageProcessor` / `AutoModel` (with two GPUs, `embeddings.run_extraction` runs one thread per device; two threads importing through transformers' lazy loader at once can raise `ImportError`), and slows tqdm to one line per 30 s so the saved outputs stay short.

In [2]:
import os, sys, json, time
os.environ.setdefault('TQDM_MININTERVAL', '30')   # before tqdm is imported
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from transformers import AutoImageProcessor, AutoModel  # noqa: F401  (import race guard)
from src.config_loader import load_config
from src import evtolnews as E
from src import registers_experiment as RX
from src import photo_crop_experiment as PX
from src import preprocess_variants as PV
from src import preprocess_controls as PC

pd.set_option('display.width', 200)
cfg = load_config()
PAT = Path(cfg['paths']['pipeline_root'])                  # 1639_LABELLED/2_embedding_extraction
EVN = E.root(cfg) / '2_embedding_extraction'               # EVTOLNEWS_DS/2_embedding_extraction
REG = RX.REG_TAG.format(size=518)
T0 = time.time()


def same_figures(csv: Path, uids) -> bool:
    # True when csv exists and lists exactly these figure_uids, in this order
    return csv.exists() and pd.read_csv(csv, usecols=['figure_uid']).figure_uid.tolist() == list(uids)


def todo(out: Path, *inputs: Path) -> bool:
    # rebuild when FORCE, when out is missing, or when an input was written after it
    return FORCE or not out.exists() or any(p.stat().st_mtime > out.stat().st_mtime for p in inputs)


def skipped(what: str, path: Path) -> None:
    print(f'[skip] {what}: on disk -> {path}')


print('FORCE          :', FORCE)
print('patent tree    :', PAT)
print('photo tree     :', EVN)
print('registers model:', RX.REG_MODEL, '->', REG)
print('devices        :', cfg['analysis'].get('devices'))

FORCE          : False
patent tree    : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction
photo tree     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/EVTOLNEWS_DS/2_embedding_extraction
registers model: facebook/dinov2-with-registers-large -> dinov2-reg-large_518
devices        : ['cuda:0', 'cuda:1']


## Step 1: registers model — embeddings, attention, artifact share
* `RX.run_extract`: `embeddings.run_extraction` with the registers model on the patent figures (`processed/518`) and on every kept photo (`EVTOLNEWS_DS/.../processed/518`), layers 18/22/24 × CLS / mean patch.
* `RX.run_attention`: CLS attention (blocks 18, 22, 24) on exactly the images of notebook 24's base attention run (the `main` set of each source).
* `RX.run_share`: per source and layer, the median and 90th percentile of the largest patch cell's share and the share spent on the prefix tokens, base vs registers. The cell also prints how many images the base and registers attention runs share: the base patent run (notebook 24, 2026-09-18) predates the 2026-09-22 refresh of the patent `main` set.

**Writes:** `1639_LABELLED/2_embedding_extraction/embeddings/dinov2-reg-large_518/`, `EVTOLNEWS_DS/2_embedding_extraction/embeddings/dinov2-reg-large_518/`, `EVTOLNEWS_DS/2_embedding_extraction/attention/dinov2-reg-large_518/{photo,patent}_{cls.npy,meta.csv}`, `.../attention/registers_share.csv`.

In [3]:
RX.run_extract(cfg, force=FORCE)
RX.run_attention(cfg, force=FORCE)
ATT = EVN / 'attention'
share_csv = ATT / 'registers_share.csv'
att_npy = [ATT / t / f'{s}_cls.npy' for t in (cfg['extraction']['tag'].format(size=518), REG) for s in ('photo', 'patent')]
if todo(share_csv, *att_npy):
    RX.run_share(cfg)
else:
    skipped('artifact-share table', share_csv)
for s in ('photo', 'patent'):   # the share table compares these image sets
    b, r = (set(pd.read_csv(ATT / t / f'{s}_meta.csv').figure_uid) for t in (cfg['extraction']['tag'].format(size=518), REG))
    print(f'{s} attention: base {len(b)} images, registers {len(r)}, common {len(b & r)}')
pd.read_csv(share_csv).round(3)

[extraction] dinov2-reg-large_518: up to date (1545 figures) — skipped
[extraction] dinov2-reg-large_518: up to date (3063 figures) — skipped


[attention] photo: up to date (1157 images)
[attention] patent: up to date (665 images)
[skip] artifact-share table: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/EVTOLNEWS_DS/2_embedding_extraction/attention/registers_share.csv
photo attention: base 1157 images, registers 1157, common 1157
patent attention: base 677 images, registers 665, common 664


,model,source,layer,n_images,max_patch_share_median,max_patch_share_p90,prefix_share_median
0,base,photo,18,1157,0.008,0.012,0.103
1,base,photo,22,1157,0.255,0.299,0.093
2,base,photo,24,1157,0.280,0.355,0.016
3,base,patent,18,677,0.007,0.012,0.116
4,base,patent,22,677,0.286,0.339,0.083
5,base,patent,24,677,0.295,0.367,0.013
6,registers,photo,18,1157,0.006,0.009,0.316
7,registers,photo,22,1157,0.015,0.026,0.280
8,registers,photo,24,1157,0.011,0.020,0.433
9,registers,patent,18,665,0.005,0.008,0.341


## Step 2: photo crop test — detect, process, extract (base and registers), montage
* `PX.run_detect`: zero-shot OWLv2 (`google/owlv2-base-patch16-ensemble`) on each hero photo of the `main` set; the best box over the aircraft prompts, padded by 10 %, becomes the crop. Below score 0.15, or a box under 1 % of the image, the full image is kept. Skipped when `crop_manifest.csv` covers the same aircraft.
* `PX.run_process`: the crops through notebook 21's resize-and-pad at 518 px.
* `PX.run_extract` (base `dinov2-large`) and `PX.run_extract_registers` (registers model): notebook 22's extraction on the crops.
* `PX.run_montage`: original | crop pairs, 8 per class, to judge the detector by eye.

The kNN scores (original vs cropped, paired bootstrap) are the evaluation step, not run here: `scripts/photo_crop_experiment.py evaluate` and `scripts/photo_crop_registers.py evaluate report`.

**Writes:** `EVTOLNEWS_DS/2_embedding_extraction/crop_experiment/` (`crops/`, `crop_manifest.csv`, `detect_summary.json`, `processed/518/`, `embeddings/dinov2-large_518/`, `embeddings/dinov2-reg-large_518/`), `EVTOLNEWS_DS/3_embedding_evaluation/crop_experiment/montage_orig_vs_crop.png`.

In [4]:
CROP = PX.exp_root(cfg)
crop_csv = CROP / 'crop_manifest.csv'
main = PX.load_main(cfg)
if FORCE or not same_figures(crop_csv, main.figure_uid):
    PX.run_detect(cfg)
else:
    skipped('OWLv2 crops', crop_csv)
crops = pd.read_csv(crop_csv, keep_default_na=False)
summ = json.loads((CROP / 'detect_summary.json').read_text())
print(f"{len(crops)} aircraft; status {summ['status']}; median crop {summ['median_crop_frac']:.2f} of the image")

proc_csv = CROP / 'processed' / str(PX.SIZE) / 'manifest.csv'
if FORCE or not same_figures(proc_csv, crops.figure_uid):
    PX.run_process(cfg, force=FORCE)
else:
    skipped('processed crops', proc_csv)
PX.run_extract(cfg, force=FORCE)
PX.run_extract_registers(cfg, force=FORCE)

montage = PX.eval_root(cfg) / 'montage_orig_vs_crop.png'
if todo(montage, crop_csv):
    PX.run_montage(cfg)
else:
    skipped('montage', montage)

[skip] OWLv2 crops: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/EVTOLNEWS_DS/2_embedding_extraction/crop_experiment/crop_manifest.csv
1157 aircraft; status {'ok': 1114, 'low_score': 24, 'tiny_box': 19}; median crop 0.64 of the image


[skip] processed crops: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/EVTOLNEWS_DS/2_embedding_extraction/crop_experiment/processed/518/manifest.csv
[extraction] dinov2-large_518: up to date (1157 figures) — skipped
[extraction] dinov2-reg-large_518: up to date (1157 figures) — skipped
[skip] montage: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/EVTOLNEWS_DS/3_embedding_evaluation/crop_experiment/montage_orig_vs_crop.png


## Step 3: preparation variants — thick1, thick2 (drawings) and grey (photos)
Per variant, in order:
* `PV.run_make`: the prepared copies of the ORIGINAL sources. `thick<k>`: greyscale, then a minimum filter of radius `round(k / s)` original pixels (`s = 518 / max(w, h)`), so each stroke side grows by about `k` px at the model input; the radius is recorded per image. `grey`: the photo to greyscale. Skipped when `sources_manifest.csv` lists the plain run's figures in the same order.
* `PV.run_process`: notebook 21's resize-and-pad at 518 px.
* `PV.run_extract`: notebook 22's extraction with the registers model, then `PV.check` (figure set and order equal the plain registers run).

Then `PV.run_montage`: plain | thick1 | thick2 as the model receives them, 8 figures of the 8 most frequent G1 classes.

**Writes:** `1639_LABELLED/2_embedding_extraction/preprocess_variants/{thick1,thick2}/` and `EVTOLNEWS_DS/2_embedding_extraction/preprocess_variants/grey/` (`sources/`, `sources_manifest.csv`, `processed/518/`, `embeddings/dinov2-reg-large_518/`), `1639_LABELLED/3_embedding_evaluation/setup_selection/montage_thickening.png`.

In [5]:
rows = []
for v in PV.VARIANTS:
    vroot = PV.variant_root(cfg, v)
    src_csv = vroot / 'sources_manifest.csv'
    if FORCE or not same_figures(src_csv, PV.plain_table(cfg, v).figure_uid):
        PV.run_make(cfg, v)
    else:
        skipped(f'{v} copies', src_csv)
    src = pd.read_csv(src_csv, keep_default_na=False)
    proc_csv = vroot / 'processed' / str(PV.SIZE) / 'manifest.csv'
    if FORCE or not same_figures(proc_csv, src.figure_uid):
        PV.run_process(cfg, v, force=FORCE)
    else:
        skipped(f'{v} processed', proc_csv)
    PV.run_extract(cfg, v, force=FORCE)
    rows.append({'variant': v, 'tree': vroot.parent.parent.parent.name, 'figures': len(src),
                 'radius counts': src.radius.value_counts().sort_index().to_dict() if 'radius' in src else '-'})

montage = PV.eval_root(cfg) / 'montage_thickening.png'
if todo(montage, *[PV.variant_root(cfg, v) / 'processed' / str(PV.SIZE) / 'manifest.csv' for v in PV.THICK]):
    PV.run_montage(cfg)
else:
    skipped('montage', montage)
pd.DataFrame(rows)

[skip] thick1 copies: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/preprocess_variants/thick1/sources_manifest.csv


[skip] thick1 processed: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/preprocess_variants/thick1/processed/518/manifest.csv


[extraction] dinov2-reg-large_518: up to date (1545 figures) — skipped
[check thick1] 1545 figures, set and order match the plain registers run


[skip] thick2 copies: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/preprocess_variants/thick2/sources_manifest.csv


[skip] thick2 processed: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/preprocess_variants/thick2/processed/518/manifest.csv

...

[skip] grey copies: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/EVTOLNEWS_DS/2_embedding_extraction/preprocess_variants/grey/sources_manifest.csv


[skip] grey processed: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/EVTOLNEWS_DS/2_embedding_extraction/preprocess_variants/grey/processed/518/manifest.csv


[extraction] dinov2-reg-large_518: up to date (1157 figures) — skipped
[check grey] 1157 figures, set and order match the plain registers run


[skip] montage: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/setup_selection/montage_thickening.png


,variant,tree,figures,radius counts
0,thick1,1639_LABELLED,1545,"{0: 5, 1: 152, 2: 266, 3: 295, 4: 479, 5: 240,..."
1,thick2,1639_LABELLED,1545,"{1: 25, 2: 112, 3: 54, 4: 195, 5: 69, 6: 136, ..."
2,grey,EVTOLNEWS_DS,1157,-


## Step 4: drawing controls — grey and contrast, ink table, montage v2
The thick variants greyscale, darken and widen at once; two controls separate the first two from the widening:
* `grey`: the drawing to greyscale, no spatial filter.
* `contrast`: made from grey's 518 px inputs (so `grey` must be processed first): per image, the ink pixels (`v < 250`) at or below their median level (clipped to 64–224) turn black. The stroke footprint stays exactly grey's: darker, not wider.

Per control: `PC.run_make` → `PC.run_process` (for contrast it asserts the 518 → 518 resize is the identity) → `PC.run_extract` (+ `PC.check`). Then `PC.run_ink` (ink mass, footprint and solid share at the model input, per preparation) and `PC.run_montage` (plain | grey | contrast | thick1 | thick2, the same 8 figures as step 3).

**Writes:** `1639_LABELLED/2_embedding_extraction/preprocess_variants/{grey,contrast}/` (`sources/`, `sources_manifest.csv`, `processed/518/`, `embeddings/dinov2-reg-large_518/`), `1639_LABELLED/3_embedding_evaluation/setup_selection/s2_controls_ink.csv`, `montage_thickening_v2.png`.

In [6]:
base = PV.plain_table(cfg, 'thick1')          # the drawing table, plain run order
for v in PC.CONTROLS:                         # grey before contrast
    vroot = PC.variant_root(cfg, v)
    src_csv = vroot / 'sources_manifest.csv'
    if FORCE or not same_figures(src_csv, base.figure_uid):
        PC.run_make(cfg, v)
    else:
        skipped(f'{v} copies', src_csv)
    proc_csv = vroot / 'processed' / str(PC.SIZE) / 'manifest.csv'
    if FORCE or not same_figures(proc_csv, base.figure_uid):
        PC.run_process(cfg, v, force=FORCE)
    else:
        skipped(f'{v} processed', proc_csv)
    PC.run_extract(cfg, v, force=FORCE)

proc5 = [PAT / 'processed' / str(PC.SIZE) / 'manifest.csv'] + \
        [PC.variant_root(cfg, v) / 'processed' / str(PC.SIZE) / 'manifest.csv' for v in PC.CONTROLS + list(PV.THICK)]
ink_csv = PV.eval_root(cfg) / 's2_controls_ink.csv'
if todo(ink_csv, *proc5):
    PC.run_ink(cfg)
else:
    skipped('ink table', ink_csv)
montage = PV.eval_root(cfg) / 'montage_thickening_v2.png'
if todo(montage, *proc5):
    PC.run_montage(cfg)
else:
    skipped('montage v2', montage)
thr = pd.read_csv(PC.variant_root(cfg, 'contrast') / 'sources_manifest.csv').threshold
print(f'contrast threshold: median {thr[thr >= 0].median():.0f}, {int((thr < 0).sum())} images left as grey')
pd.read_csv(ink_csv).round(4)

[skip] grey copies: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/preprocess_variants/grey/sources_manifest.csv


[skip] grey processed: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/preprocess_variants/grey/processed/518/manifest.csv
[extraction] dinov2-reg-large_518: up to date (1545 figures) — skipped


[check grey] 1545 figures, set and order match the plain registers run


[skip] contrast copies: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/preprocess_variants/contrast/sources_manifest.csv
[skip] contrast processed: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/preprocess_variants/contrast/processed/518/manifest.csv


[extraction] dinov2-reg-large_518: up to date (1545 figures) — skipped
[check contrast] 1545 figures, set and order match the plain registers run


[skip] ink table: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/setup_selection/s2_controls_ink.csv
[skip] montage v2: on disk -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/setup_selection/montage_thickening_v2.png
contrast threshold: median 148, 0 images left as grey


,preparation,n_figures,ink_mass_mean,footprint_mean,solid_share_mean
0,plain,1545,0.0336,0.0819,0.0300
1,grey,1545,0.0336,0.0818,0.0300
2,contrast,1545,0.0481,0.0818,0.0419
3,thick1,1545,0.0998,0.1377,0.0994
4,thick2,1545,0.1393,0.1731,0.1395


## Outputs
Every path this notebook names, checked on disk. The embedding folders must hold `metadata.parquet` and the six `emb_layer*.npy` matrices.

In [7]:
EVAL_PAT = PV.eval_root(cfg)
outputs = {
    'patent registers embeddings': PAT / 'embeddings' / REG,
    'photo registers embeddings': EVN / 'embeddings' / REG,
    'registers attention (photo)': EVN / 'attention' / REG / 'photo_cls.npy',
    'registers attention (patent)': EVN / 'attention' / REG / 'patent_cls.npy',
    'artifact-share table': EVN / 'attention' / 'registers_share.csv',
    'crop manifest': CROP / 'crop_manifest.csv',
    'detect summary': CROP / 'detect_summary.json',
    'processed crops': CROP / 'processed' / '518' / 'manifest.csv',
    'crop embeddings (base)': CROP / 'embeddings' / cfg['extraction']['tag'].format(size=518),
    'crop embeddings (registers)': CROP / 'embeddings' / REG,
    'crop montage': PX.eval_root(cfg) / 'montage_orig_vs_crop.png',
    **{f'variant {v} embeddings': PV.variant_root(cfg, v) / 'embeddings' / REG for v in PV.VARIANTS},
    'thickening montage': EVAL_PAT / 'montage_thickening.png',
    **{f'control {v} embeddings': PC.variant_root(cfg, v) / 'embeddings' / REG for v in PC.CONTROLS},
    'ink table': EVAL_PAT / 's2_controls_ink.csv',
    'thickening montage v2': EVAL_PAT / 'montage_thickening_v2.png',
}
status = {}
for name, p in outputs.items():
    ok = p.exists() and (not p.is_dir() or ((p / 'metadata.parquet').exists() and len(list(p.glob('emb_layer*.npy'))) == 6))
    status[name] = {'ok': ok, 'rows': len(pd.read_parquet(p / 'metadata.parquet')) if ok and p.is_dir() else '',
                    'path': str(p).split('Labelling Outputs/')[-1]}
status = pd.DataFrame(status).T
assert status.ok.all(), status[~status.ok]
print(f'all {len(status)} outputs on disk; steps 1-4 took {time.time() - T0:.1f} s')
status

all 19 outputs on disk; steps 1-4 took 1.5 s


,ok,rows,path
patent registers embeddings,True,1545,1639_LABELLED/2_embedding_extraction/embedding...
photo registers embeddings,True,3063,EVTOLNEWS_DS/2_embedding_extraction/embeddings...
registers attention (photo),True,,EVTOLNEWS_DS/2_embedding_extraction/attention/...
registers attention (patent),True,,EVTOLNEWS_DS/2_embedding_extraction/attention/...
artifact-share table,True,,EVTOLNEWS_DS/2_embedding_extraction/attention/...
crop manifest,True,,EVTOLNEWS_DS/2_embedding_extraction/crop_exper...
detect summary,True,,EVTOLNEWS_DS/2_embedding_extraction/crop_exper...
processed crops,True,,EVTOLNEWS_DS/2_embedding_extraction/crop_exper...
crop embeddings (base),True,1157,EVTOLNEWS_DS/2_embedding_extraction/crop_exper...
crop embeddings (registers),True,1157,EVTOLNEWS_DS/2_embedding_extraction/crop_exper...
